# CorticalClock

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.CorticalClock)

class CorticalClock(LinearReferenceClock):
    def postprocess(self, x):
        """Horvath anti-logarithmic linear transformation (adult age = 20)."""
        adult_age = 20
        mask_negative = x < 0
        mask_non_negative = ~mask_negative
        age = torch.empty_like(x)
        age[mask_negative] = (1 + adult_age) * torch.exp(x[mask_negative]) - 1
        age[mask_non_negative] = (1 + adult_age) * x[mask_non_negative] + adult_age
        return age



In [3]:
model = pya.models.CorticalClock()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "corticalclock"
model.metadata["data_type"] = "DNA methylation"  # Paper: methylation
model.metadata["species"] = "Homo sapiens"  # Paper: Homo sapiens
model.metadata["year"] = 2020
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Shireby, G. L., et al. \"Recalibrating the epigenetic clock: implications for assessing biological age in the human cortex.\" Brain 143.12 (2020): 3763-3775."
model.metadata["doi"] = "https://doi.org/10.1093/brain/awaa334"
model.metadata["notes"] = "Cortex-specific DNA-methylation chronological-age estimator trained by elastic net on 1,047 post-mortem cortical samples; its 347-CpG weighted score is back-transformed to years."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["brain cortex"]  # Paper: brain cortex (post-mortem human cortical tissue)
model.metadata["predicts"] = ["chronological age"]  # Paper: chronological age
model.metadata["training_target"] = ["chronological age"]  # Paper: chronological age
model.metadata["unit"] = ["years"]  # Paper: years
model.metadata["model_type"] = "elastic net regression"  # Paper: Elastic net
model.metadata["platform"] = ["Illumina 450K"]  # Paper: Illumina 450K
model.metadata["population"] = "human, age unspecified"  # Paper: human cortex training set: 1,047 samples from 832 donors, ages 1–108 years
model.metadata["journal"] = "Brain"
model.metadata["last_author"] = "Jonathan Mill"
model.metadata["n_features"] = 347
model.metadata["citations"] = 206
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o CorticalClockCoefs.txt https://raw.githubusercontent.com/gemmashireby/CorticalClock/80c3df19c01d9aac25c9fd5aecd5bbc42aba0939/PredCorticalAge/CorticalClockCoefs.txt")
os.system(f"curl -sL -o Ref_DNAm_brain_values.rdat https://raw.githubusercontent.com/gemmashireby/CorticalClock/80c3df19c01d9aac25c9fd5aecd5bbc42aba0939/PredCorticalAge/Ref_DNAm_brain_values.rdat")

0

In [6]:
%%writefile download.r

library(jsonlite)
coefs <- read.table("CorticalClockCoefs.txt", header = TRUE, stringsAsFactors = FALSE)
coefs <- coefs[tolower(coefs$probe) != "intercept" & tolower(coefs$probe) != "(intercept)", ]
load("Ref_DNAm_brain_values.rdat")
refvals <- as.numeric(ref[coefs$probe])
write_json(list(probe = coefs$probe, coef = coefs$coef, ref = refvals), "cortical.json", digits = 10)

Writing download.r


In [7]:
os.system("Rscript download.r")

0

## Load features

In [8]:
d = json.load(open('cortical.json'))
model.features = list(d['probe'])

## Load weights into base model

In [9]:
# Intercept 0.577682570446177 hard-coded in the CorticalClock reference script
weights = torch.tensor(d['coef']).unsqueeze(0).float()
intercept = torch.tensor([0.577682570446177]).float()

In [10]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [11]:
model.reference_values = d['ref']

## Load preprocess and postprocess objects

In [12]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [13]:
model.postprocess_name = 'anti_log_linear'
model.postprocess_dependencies = None

## Check all clock parameters

In [14]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Shireby, Gemma L., et al. "Recalibrating the epigenetic clock: '
             'implications for assessing biological age in the human cortex." '
             'Brain 143.12 (2020): 3763-3775.',
 'clock_name': 'corticalclock',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1093/brain/awaa334',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2020}
reference_values: [0.21611617687, 0.15740657645, 0.7341710846, 0.060822631032, 0.10575749654, 0.62440019335, 0.13956151931, 0.93910437046, 0.67922573527, 0.41269620156, 0.18058134103, 0.30387161589, 0.72916401957, 0.16249365613, 0.10514838761, 0.32006057604, 0.41533755409, 0.76224987996, 0.1012865392, 0.068627506888, 0.62711858238, 0.067610974155, 0.73657937789, 0.085680987307, 0.16091221649, 0.5857097145, 0.6569132

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [16]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [17]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: CorticalClockCoefs.txt
Deleted file: Ref_DNAm_brain_values.rdat
Deleted file: download.r
Deleted file: cortical.json
